# Week 3: RL Trading Dashboard

Interactive dashboard comparing PPO agent vs baseline strategies.

**Data Sources:**
- `market_trace.csv` - Multi-agent simulation data
- `ppo_agent_log.csv` - PPO evaluation results
- `random_agent_log.csv` - Random agent baseline
- `buyhold_agent_log.csv` - Buy & Hold baseline

**Charts:**
1. Portfolio Value Comparison (PPO vs Buy&Hold vs Random)
2. Price Chart with PPO BUY/SELL Markers
3. PnL Distribution Histogram

In [13]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

# Configure plotly for VS Code notebooks - use HTML renderer only
pio.renderers.default = 'notebook_connected'
pio.renderers.render_on_display = False

In [8]:
# Define data paths - use absolute paths
PROJECT_ROOT = r'c:\Users\HP\Citadel--Algo-WIDS'
TRAINING_DIR = os.path.join(PROJECT_ROOT, 'week3', 'training')
ANALYSIS_DIR = os.path.join(PROJECT_ROOT, 'week3', 'analysis')

# Load all data
print("Loading data...")

# Market trace data
market_trace = pd.read_csv(os.path.join(ANALYSIS_DIR, 'market_trace.csv'))
print(f"Market trace: {len(market_trace)} rows")

# Agent evaluation logs
ppo_log = pd.read_csv(os.path.join(TRAINING_DIR, 'ppo_agent_log.csv'))
random_log = pd.read_csv(os.path.join(TRAINING_DIR, 'random_agent_log.csv'))
buyhold_log = pd.read_csv(os.path.join(TRAINING_DIR, 'buyhold_agent_log.csv'))

print(f"PPO log: {len(ppo_log)} rows")
print(f"Random log: {len(random_log)} rows")
print(f"Buy&Hold log: {len(buyhold_log)} rows")

# Training log
training_log = pd.read_csv(os.path.join(TRAINING_DIR, 'training_log.csv'))
print(f"Training log: {len(training_log)} rows")

Loading data...
Market trace: 93454 rows
PPO log: 10000 rows
Random log: 10000 rows
Buy&Hold log: 10000 rows
Training log: 25 rows


## Chart 1: Portfolio Value Comparison

Compare portfolio values over time for:
- **PPO Agent** (trained RL)
- **Buy & Hold** (passive baseline)
- **Random Agent** (random actions)

In [14]:
# Aggregate portfolio values by step (mean across episodes)
ppo_agg = ppo_log.groupby('step')['portfolio_value'].mean().reset_index()
ppo_agg['strategy'] = 'PPO Agent'

random_agg = random_log.groupby('step')['portfolio_value'].mean().reset_index()
random_agg['strategy'] = 'Random Agent'

buyhold_agg = buyhold_log.groupby('step')['portfolio_value'].mean().reset_index()
buyhold_agg['strategy'] = 'Buy & Hold'

# Combine all strategies
portfolio_df = pd.concat([ppo_agg, random_agg, buyhold_agg], ignore_index=True)

# Create interactive line chart
fig1 = px.line(
    portfolio_df,
    x='step',
    y='portfolio_value',
    color='strategy',
    title='📈 Portfolio Value: PPO vs Buy&Hold vs Random',
    labels={'step': 'Time Step', 'portfolio_value': 'Portfolio Value ($)', 'strategy': 'Strategy'},
    color_discrete_map={
        'PPO Agent': '#2ecc71',
        'Buy & Hold': '#3498db',
        'Random Agent': '#e74c3c'
    }
)

fig1.update_layout(
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500
)

fig1.update_traces(hovertemplate='%{y:$.2f}')

# Display using HTML
from IPython.display import HTML
HTML(fig1.to_html(include_plotlyjs='cdn'))

## Chart 2: Price with PPO BUY/SELL Markers

Visualize the mid-price over time with PPO agent's trade decisions overlaid.

In [15]:
# Get PPO trades from market trace
ppo_trades = market_trace[market_trace['agent_id'] == 'PPO'].copy()

# Get unique mid prices over time (one per timestamp)
price_series = market_trace.groupby('timestamp')['mid_price'].first().reset_index()

# Separate BUY and SELL actions
ppo_buys = ppo_trades[ppo_trades['action'] == 'BUY']
ppo_sells = ppo_trades[ppo_trades['action'] == 'SELL']

print(f"PPO trades: {len(ppo_trades)} total ({len(ppo_buys)} BUY, {len(ppo_sells)} SELL)")

# Create figure with price line and trade markers
fig2 = go.Figure()

# Price line
fig2.add_trace(go.Scatter(
    x=price_series['timestamp'],
    y=price_series['mid_price'],
    mode='lines',
    name='Mid Price',
    line=dict(color='#3498db', width=1.5),
    hovertemplate='Time: %{x:.1f}s<br>Price: $%{y:.2f}<extra></extra>'
))

# BUY markers
fig2.add_trace(go.Scatter(
    x=ppo_buys['timestamp'],
    y=ppo_buys['mid_price'],
    mode='markers',
    name='PPO BUY',
    marker=dict(color='#2ecc71', size=10, symbol='triangle-up'),
    hovertemplate='BUY @ %{y:.2f}<br>Time: %{x:.1f}s<extra></extra>'
))

# SELL markers
fig2.add_trace(go.Scatter(
    x=ppo_sells['timestamp'],
    y=ppo_sells['mid_price'],
    mode='markers',
    name='PPO SELL',
    marker=dict(color='#e74c3c', size=10, symbol='triangle-down'),
    hovertemplate='SELL @ %{y:.2f}<br>Time: %{x:.1f}s<extra></extra>'
))

fig2.update_layout(
    title='📊 Price Chart with PPO Agent BUY/SELL Markers',
    xaxis_title='Time (seconds)',
    yaxis_title='Mid Price ($)',
    template='plotly_dark',
    hovermode='closest',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500
)

# Enable range slider for zooming
fig2.update_xaxes(rangeslider_visible=True)

HTML(fig2.to_html(include_plotlyjs='cdn'))

PPO trades: 5000 total (0 BUY, 5000 SELL)


## Chart 3: PnL Histogram

Distribution of per-episode final PnL (Profit & Loss) for each strategy.

In [16]:
# Calculate final PnL for each episode
initial_value = 10000.0

def get_final_pnl(df):
    """Get final PnL for each episode."""
    final_values = df.groupby('episode')['portfolio_value'].last()
    return final_values - initial_value

ppo_pnl = get_final_pnl(ppo_log)
random_pnl = get_final_pnl(random_log)
buyhold_pnl = get_final_pnl(buyhold_log)

# Create histogram data
pnl_data = pd.DataFrame({
    'PnL': list(ppo_pnl) + list(random_pnl) + list(buyhold_pnl),
    'Strategy': ['PPO Agent'] * len(ppo_pnl) + ['Random Agent'] * len(random_pnl) + ['Buy & Hold'] * len(buyhold_pnl)
})

# Create grouped histogram
fig3 = px.histogram(
    pnl_data,
    x='PnL',
    color='Strategy',
    title='📉 PnL Distribution by Strategy',
    labels={'PnL': 'Profit & Loss ($)', 'count': 'Frequency'},
    color_discrete_map={
        'PPO Agent': '#2ecc71',
        'Buy & Hold': '#3498db',
        'Random Agent': '#e74c3c'
    },
    barmode='group',
    nbins=20
)

# Add vertical line at 0
fig3.add_vline(x=0, line_dash='dash', line_color='white', annotation_text='Break Even')

fig3.update_layout(
    template='plotly_dark',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=400
)

# Print summary statistics
print("📊 PnL Summary Statistics:")
print(f"{'Strategy':<15} {'Mean PnL':>12} {'Std Dev':>12} {'Min':>12} {'Max':>12}")
print("-" * 65)
for strategy, pnl in [('PPO Agent', ppo_pnl), ('Random Agent', random_pnl), ('Buy & Hold', buyhold_pnl)]:
    print(f"{strategy:<15} ${pnl.mean():>10.2f} ${pnl.std():>10.2f} ${pnl.min():>10.2f} ${pnl.max():>10.2f}")

HTML(fig3.to_html(include_plotlyjs='cdn'))

📊 PnL Summary Statistics:
Strategy            Mean PnL      Std Dev          Min          Max
-----------------------------------------------------------------
PPO Agent       $      0.00 $      0.00 $      0.00 $      0.00
Random Agent    $  -3544.70 $    150.67 $  -3736.33 $  -3375.88
Buy & Hold      $     -0.50 $      0.03 $     -0.54 $     -0.46


## Bonus: Training Progress

Visualize PPO training metrics over time.

In [17]:
# Create training progress subplot
fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Mean Episode Reward', 'Action Distribution', 'Entropy', 'Cumulative Actions'),
    specs=[[{}, {}], [{}, {}]]
)

# 1. Mean Episode Reward
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['mean_episode_reward'],
        mode='lines+markers',
        name='Mean Reward',
        line=dict(color='#2ecc71'),
        hovertemplate='Step: %{x}<br>Reward: %{y:.0f}<extra></extra>'
    ),
    row=1, col=1
)

# 2. Action Distribution (stacked area)
total_actions = training_log['action_hold'] + training_log['action_buy'] + training_log['action_sell']
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_hold'] / total_actions * 100,
        mode='lines',
        name='HOLD %',
        line=dict(color='#f39c12'),
        stackgroup='actions'
    ),
    row=1, col=2
)
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_buy'] / total_actions * 100,
        mode='lines',
        name='BUY %',
        line=dict(color='#2ecc71'),
        stackgroup='actions'
    ),
    row=1, col=2
)
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_sell'] / total_actions * 100,
        mode='lines',
        name='SELL %',
        line=dict(color='#e74c3c'),
        stackgroup='actions'
    ),
    row=1, col=2
)

# 3. Entropy
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['entropy'],
        mode='lines',
        name='Entropy',
        line=dict(color='#9b59b6'),
        hovertemplate='Step: %{x}<br>Entropy: %{y:.4f}<extra></extra>'
    ),
    row=2, col=1
)

# 4. Cumulative Actions
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_hold'],
        mode='lines',
        name='Total HOLD',
        line=dict(color='#f39c12', dash='solid')
    ),
    row=2, col=2
)
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_buy'],
        mode='lines',
        name='Total BUY',
        line=dict(color='#2ecc71', dash='solid')
    ),
    row=2, col=2
)
fig4.add_trace(
    go.Scatter(
        x=training_log['timestep'],
        y=training_log['action_sell'],
        mode='lines',
        name='Total SELL',
        line=dict(color='#e74c3c', dash='solid')
    ),
    row=2, col=2
)

fig4.update_layout(
    title='🎓 PPO Training Progress',
    template='plotly_dark',
    height=600,
    showlegend=True
)

HTML(fig4.to_html(include_plotlyjs='cdn'))

## Summary

This dashboard provides an interactive view of:

1. **Portfolio Comparison** - PPO agent vs baselines
2. **Trade Visualization** - Where PPO decides to buy/sell
3. **PnL Distribution** - Risk/reward profile of each strategy
4. **Training Progress** - How PPO learned over time

**Interactive Features:**
- 🔍 **Zoom**: Click and drag to zoom, double-click to reset
- 👆 **Hover**: Hover over points for detailed values
- 👁️ **Toggle**: Click legend items to show/hide traces
- 📏 **Range Slider**: Use the slider below Chart 2 to focus on specific time ranges